# associate rule

# For more info click the link below

https://docs.google.com/document/d/1IDdBw7GpcX_Y2_NP7OcqfkZrj4q_QYMfUYzwKM73Um8/edit?usp=sharing

# notes and interview Q and A

https://docs.google.com/document/d/1_2BPsulahheVbg0XC8TJRjF6J1ukBVU9o95tkdHtUDk/edit?usp=sharing

# coding

# Import libraries 

In [1]:
import pandas as pd                                      # Data manipulation

from mlxtend.preprocessing import TransactionEncoder    # Convert transactions into one-hot format
from mlxtend.frequent_patterns import fpgrowth          # Find frequent itemsets
from mlxtend.frequent_patterns import association_rules # Generate association rules

# Create transaction dataset 

In [2]:
transactions = [
    ["Milk", "Bread", "Butter"],                         # Transaction 1
    ["Milk", "Bread"],                                   # Transaction 2
    ["Milk", "Bread", "Butter", "Eggs"],                 # Transaction 3
    ["Bread", "Butter"],                                 # Transaction 4
    ["Milk", "Bread", "Eggs"],                           # Transaction 5
    ["Milk", "Butter"],                                  # Transaction 6
    ["Bread", "Butter", "Eggs"],                         # Transaction 7
    ["Milk", "Bread", "Butter"],                         # Transaction 8
    ["Milk", "Bread", "Eggs"],                           # Transaction 9
    ["Bread", "Butter"]                                  # Transaction 10
]

# Convert transaction to one hot encoding 

In [3]:
encoder = TransactionEncoder()                           # Create an encoder for transaction data

encoded_data = encoder.fit(
    transactions                                             # Learn all unique products
).transform(
    transactions                                             # Convert each transaction into True/False values
)

df = pd.DataFrame(
    encoded_data,                                         # Store encoded transaction data
    columns=encoder.columns_                              # Use product names as column names
)

print(df)

   Bread  Butter   Eggs   Milk
0   True    True  False   True
1   True   False  False   True
2   True    True   True   True
3   True    True  False  False
4   True   False   True   True
5  False    True  False   True
6   True    True   True  False
7   True    True  False   True
8   True   False   True   True
9   True    True  False  False


# Find frequent items 


In [4]:
frequent_itemsets = fpgrowth(
    df,                                                    # Transaction dataset
    min_support=0.3,                                      # Itemset must appear in at least 30% of transactions
    use_colnames=True                                     # Show actual product names
)

print(frequent_itemsets)

    support               itemsets
0       0.9                (Bread)
1       0.7                 (Milk)
2       0.7               (Butter)
3       0.4                 (Eggs)
4       0.6          (Bread, Milk)
5       0.6        (Bread, Butter)
6       0.4         (Butter, Milk)
7       0.3  (Bread, Butter, Milk)
8       0.4          (Bread, Eggs)
9       0.3           (Milk, Eggs)
10      0.3    (Bread, Milk, Eggs)


# Generate association rules 

In [5]:
rules = association_rules(
    frequent_itemsets,                                    # Frequent itemsets discovered by FP-Growth
    metric="confidence",                                  # Use confidence to evaluate rules
    min_threshold=0.6                                    # Keep rules with at least 60% confidence
)

print(rules)

      antecedents    consequents  antecedent support  consequent support  \
0         (Bread)         (Milk)                 0.9                 0.7   
1          (Milk)        (Bread)                 0.7                 0.9   
2         (Bread)       (Butter)                 0.9                 0.7   
3        (Butter)        (Bread)                 0.7                 0.9   
4  (Butter, Milk)        (Bread)                 0.4                 0.9   
5          (Eggs)        (Bread)                 0.4                 0.9   
6          (Eggs)         (Milk)                 0.4                 0.7   
7   (Bread, Eggs)         (Milk)                 0.4                 0.7   
8    (Milk, Eggs)        (Bread)                 0.3                 0.9   
9          (Eggs)  (Bread, Milk)                 0.4                 0.6   

   support  confidence      lift  representativity  leverage  conviction  \
0      0.6    0.666667  0.952381               1.0     -0.03         0.9   
1      0.6 

# Select the important columns 

In [6]:
rules = rules[
    [
        "antecedents",                                    # Items on the left side of the rule
        "consequents",                                    # Items on the right side of the rule
        "support",                                        # Frequency of the complete itemset
        "confidence",                                     # Probability of consequent given antecedent
        "lift"                                            # Strength of association
    ]
]

print(rules)

      antecedents    consequents  support  confidence      lift
0         (Bread)         (Milk)      0.6    0.666667  0.952381
1          (Milk)        (Bread)      0.6    0.857143  0.952381
2         (Bread)       (Butter)      0.6    0.666667  0.952381
3        (Butter)        (Bread)      0.6    0.857143  0.952381
4  (Butter, Milk)        (Bread)      0.3    0.750000  0.833333
5          (Eggs)        (Bread)      0.4    1.000000  1.111111
6          (Eggs)         (Milk)      0.3    0.750000  1.071429
7   (Bread, Eggs)         (Milk)      0.3    0.750000  1.071429
8    (Milk, Eggs)        (Bread)      0.3    1.000000  1.111111
9          (Eggs)  (Bread, Milk)      0.3    0.750000  1.250000


# Filter rules using confidence 

`let's keep only rules with confidence >= 70% `

In [7]:
strong_rules = rules[
    rules["confidence"] >= 0.70                          # Keep rules with at least 70% confidence
]

print(strong_rules)

      antecedents    consequents  support  confidence      lift
1          (Milk)        (Bread)      0.6    0.857143  0.952381
3        (Butter)        (Bread)      0.6    0.857143  0.952381
4  (Butter, Milk)        (Bread)      0.3    0.750000  0.833333
5          (Eggs)        (Bread)      0.4    1.000000  1.111111
6          (Eggs)         (Milk)      0.3    0.750000  1.071429
7   (Bread, Eggs)         (Milk)      0.3    0.750000  1.071429
8    (Milk, Eggs)        (Bread)      0.3    1.000000  1.111111
9          (Eggs)  (Bread, Milk)      0.3    0.750000  1.250000


# Find the strongest rules using lift 

In [8]:
strong_rules = strong_rules.sort_values(
    by="lift",                                           # Sort according to association strength
    ascending=False                                      # Show strongest rules first
)

print(strong_rules)

      antecedents    consequents  support  confidence      lift
9          (Eggs)  (Bread, Milk)      0.3    0.750000  1.250000
5          (Eggs)        (Bread)      0.4    1.000000  1.111111
8    (Milk, Eggs)        (Bread)      0.3    1.000000  1.111111
6          (Eggs)         (Milk)      0.3    0.750000  1.071429
7   (Bread, Eggs)         (Milk)      0.3    0.750000  1.071429
1          (Milk)        (Bread)      0.6    0.857143  0.952381
3        (Butter)        (Bread)      0.6    0.857143  0.952381
4  (Butter, Milk)        (Bread)      0.3    0.750000  0.833333


# Experiment with the hyperparameters 

`There are 3 main important things to experiment with `

`min_support `
`Lower support`

`min_support`
`more frequent itemsets`

`High support `
`↑ min_support`
`↓`
`Fewer frequent itemsets`

`min_threshold=0.5`

In [9]:
for confidence_threshold in [0.4, 0.5, 0.6, 0.7, 0.8]:

    rules = association_rules(
        frequent_itemsets,                                # Use frequent itemsets from FP-Growth
        metric="confidence",                              # Evaluate rules using confidence
        min_threshold=confidence_threshold               # Change minimum confidence requirement
    )

    print(
        f"Confidence ≥ {confidence_threshold}: "
        f"{len(rules)} rules"
    )

Confidence ≥ 0.4: 20 rules
Confidence ≥ 0.5: 15 rules
Confidence ≥ 0.6: 10 rules
Confidence ≥ 0.7: 8 rules
Confidence ≥ 0.8: 4 rules
